# Mol-1 structure prediction on Colab

1. **Runtime → Change runtime type → GPU** (T4 is enough for this 346-residue protein).
2. Run the cells top to bottom. The install and the first prediction take a while: the first run also compiles a CUDA LayerNorm extension (a few minutes of compiler output that looks like a hang).

In [ ]:
#@title 1. Check GPU and Python
import subprocess, sys, re
print(sys.version)
assert sys.version_info >= (3, 12), "Mol-1 needs Python 3.12+"
smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
print(smi)
assert smi, "No GPU: Runtime -> Change runtime type -> GPU"
cuda_major = int(re.search(r"CUDA Version:\s*(\d+)", smi).group(1))
EXTRA = "cuda" if cuda_major >= 13 else "cuda12"
print("Driver supports CUDA", cuda_major, "-> installing extra:", EXTRA)

def run(cmd):
    """Run a shell command, stream its output, and stop the notebook if it fails."""
    p = subprocess.run(cmd, shell=True)
    if p.returncode != 0:
        raise RuntimeError(f"Command failed (exit {p.returncode}): {cmd}")

In [ ]:
#@title 2. Install Mol-1 (~5 min)
import os
os.chdir("/content")
if not os.path.isdir("Mol-1"):
    run("git clone -q https://github.com/molence/Mol-1")
os.chdir("/content/Mol-1")
run(f'pip install -q -e ".[{EXTRA}]"')
run("pip install -q ninja")  # builds the fused LayerNorm CUDA extension; Colab lacks it
run("mol doctor")

In [ ]:
#@title 3. Download reference data (~0.3 GB) and weights (~2.2 GB)
import os
os.environ["MOL_ROOT_DIR"] = "/content/mol_data"
run("bash scripts/database/download_mol_data.sh")
run("mkdir -p $MOL_ROOT_DIR/checkpoint")
run("wget -q -c https://mol-1-release.oss-cn-shanghai.aliyuncs.com/checkpoint/mol-1-base.pt -O $MOL_ROOT_DIR/checkpoint/mol-1-base.pt")

In [ ]:
#@title 4. Input
import json
NAME = "my_protein"
SEQUENCE = """
MKYWQKHAVLCSLLVGASLWILPQADAAKAPEQTVTEPVGSYARAERPQDFEGFVWRLDN
DGKEALPRNFRTSADALRAPEKKFHLDAAYVPSREGMDALHISGSSAFTPAQLKNVAAKL
REKTAGPIYDVDLRQESHGYLDGIPVSWYGERDWANLGKSQHEALADERHRLHAALHKTV
YIAPLGKHKLPEGGEVRRVQKVQTEQEVAEAAGMRYFRIAATDHVWPTPENIDRFLAFYR
TLPQDAWLHFHCEAGVGRTTAFMVMTDMLKNPSVSLKDILYRQHEIGGFYYGEFPIKTKD
KDSWKTKYYREKIVMIEQFYRYVQENRADGYQTPWSVWLKSHPAKA
"""
seq = "".join(SEQUENCE.split())
print(len(seq), "residues")
job = [{"name": NAME, "modelSeeds": [101],
        "sequences": [{"proteinChain": {"sequence": seq, "count": 1}}]}]
json.dump(job, open(f"/content/{NAME}.json", "w"), indent=2)

In [ ]:
#@title 5a. MSA search on the public ColabFold server, retried while it is down
import time, os
msa_json = f"/content/{NAME}-update-msa.json"
for attempt in range(1, 11):
    p = subprocess.run(f"mol msa -i /content/{NAME}.json -o /content/msa", shell=True)
    if p.returncode == 0 and os.path.exists(msa_json):
        print("MSA ready:", msa_json)
        break
    print(f"MSA server failed (attempt {attempt}/10); retrying in 3 minutes")
    time.sleep(180)
else:
    raise RuntimeError("ColabFold MSA server still failing; try again later")

In [ ]:
#@title 5b. Predict (5 samples)
import torch
if torch.cuda.get_device_capability()[0] < 8:
    # Mol-1's fused LayerNorm is not built for pre-Ampere GPUs such as the T4;
    # use PyTorch's LayerNorm there.
    os.environ["LAYERNORM_TYPE"] = "torch"
run("rm -rf /content/output/ERR")  # so any report there is from this run
# Output is written to /content/pred.log as well, so errors are not lost.
p = subprocess.run(f"mol pred -i {msa_json} -o /content/output --need_atom_confidence true 2>&1 | tee /content/pred.log; exit ${{PIPESTATUS[0]}}",
                   shell=True, executable="/bin/bash")
if p.returncode != 0:
    run("ls -R /content/output/ERR 2>/dev/null; cat /content/output/ERR/* 2>/dev/null | tail -50")
    raise RuntimeError("mol pred failed; see the log above or /content/pred.log")

In [ ]:
#@title 6. Confidence summary
import glob
pred_dir = f"/content/output/{NAME}/seed_101/predictions"
for f in sorted(glob.glob(f"{pred_dir}/*_summary_confidence_sample_*.json")):
    c = json.load(open(f))
    print(f.split("/")[-1], f"pLDDT={c['plddt']:.1f}  pTM={c['ptm']:.3f}  ranking={c['ranking_score']:.3f}")

In [ ]:
#@title 7. View the top sample, coloured by pLDDT (blue = confident, red = low)
!pip install -q py3Dmol
import py3Dmol
cif = open(f"{pred_dir}/{NAME}_sample_0.cif").read()
v = py3Dmol.view(width=800, height=600)
v.addModel(cif, "cif")
v.setStyle({"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb", "min": 50, "max": 90}}})
v.zoomTo(); v.show()

In [ ]:
#@title 8. Download results
!cd /content && zip -qr {NAME}_results.zip output/{NAME}
from google.colab import files
files.download(f"/content/{NAME}_results.zip")